## 1.4. Reseñas y Manuales → Base de datos vectorial
#### Objetivo: Extraer vectores a partir de la información dada y almacenar en una base de datos vectorial.

## Pasos sugeridos
1. Leer todos los archivos de texto de resenas_usuarios/ y manuales_productos/.
2. Inicializar un divisor de texto en base a criterios propios (300 caracteres de longitud y 50 de overlap por ejemplo).
3. Fragmentar estos textos.
4. Inicializar un modelo de embeddings que soporte español (por ejemplo intfloat/multilingual-e5-small).
5. Vectorizar esos fragmentos de texto.
6. Inicializar alguna base de datos vectorial (por ejemplo, ChromaDB).
7. Almacenar esos fragmentos de texto vectorizados en la base de datos vectorial

IMPORTANTE: recordar almacenar Metadata a cada vector

## Librerias

In [1]:
import json
import os
from pymongo import MongoClient
import pandas as pd
import tensorflow_text
import tensorflow_hub as hub
from chromadb.config import Settings
import chromadb

2026-06-04 11:44:01.155377: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-04 11:44:02.355619: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-04 11:44:08.949296: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/gcasado0/miniconda3/lib/python3.13/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


## 1. Carga de Reseñas en base vectorial 

### Vectorizar las reseñas

In [5]:
# recorrer las resenias de los usuarios almacenarlas en mongoDB
# conectar a mongoDB

client = MongoClient("mongodb://localhost:27017/")
db = client["reseñas_productos"]
collection_mongo_db = db["reseñas"]


In [6]:
for registro in collection_mongo_db.find():
    print(registro)

{'_id': ObjectId('6a1619a8fb7daa06050f55f1'), 'fecha': '2024-10-25 07:45', 'usuario': 'Patricia_Ruiz', 'telefono': '+54 9 11 8901-2345', 'producto_nombre': 'Cafetera Negra', 'producto_id': None, 'puntaje': 5, 'comentario': 'La cafetera es UN SUEÑO! El molinillo integrado es lo mejor. El café queda con un aroma increíble, mucho mejor que con café ya molido. El temporizador es perfecto, programo la noche anterior y me despierto con el café listo. Vale cada peso!', 'sentimiento': 'positivo', 'aspectos_positivos': ['molinillo integrado', 'aroma del café', 'temporizador', 'relación calidad-precio'], 'aspectos_negativos': [], 'id_resenia': 'resena_007.txt'}
{'_id': ObjectId('6a1619aefb7daa06050f55f2'), 'fecha': '2024-10-13 16:20', 'usuario': 'Romina_Calderon', 'telefono': '+54 9 11 4567-8909', 'producto_nombre': 'ventilador de torre', 'producto_id': 'P019', 'puntaje': 4, 'comentario': 'El ventilador de torre es lindo y funciona bien. Refresca toda la habitación. El control remoto es práctico

In [2]:
# Inicializar un modelo de embeddings que soporte español (por ejemplo intfloat/multilingual-e5-small).

# Cargar Universal Sentence Encoder
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder-multilingual/3")

# Configuración inicial de ChromaDB
settings = Settings(
    is_persistent=True,
    persist_directory="data/chroma"   # carpeta donde se guardan los datos
)

client = chromadb.Client(settings=settings)
collection = client.get_or_create_collection("u5_practica")

2026-06-04 11:45:28.102438: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
# Cargo la base vectorial con las resenias de los usuarios almacenadas en mongoDB

for registro in collection_mongo_db.find():
    texto_id = registro['id_resenia']
    texto = registro['comentario']
    # Convertir registro['producto_id'] a string si no lo es y si es None asignarle un valor por defecto
    if not isinstance(registro['producto_id'], str):
        if registro['producto_id'] is None:
            registro['producto_id'] = ""
        else:
            registro['producto_id'] = str(registro['producto_id'])
    # Convertir registro['producto_nombre'] a string si no lo es y si es None asignarle un valor por defecto
    if not isinstance(registro['producto_nombre'], str):
        if registro['producto_nombre'] is None:
            registro['producto_nombre'] = ""
        else:
            registro['producto_nombre'] = str(registro['producto_nombre'])

    metadata = {"producto_nombre": registro['producto_nombre'], "producto_id": registro['producto_id'], "puntaje": registro['puntaje'], "sentimiento": registro['sentimiento']}
    print(f"Agregando texto con ID: {texto_id} y metadata: {metadata}")
    embedding = embed([texto]).numpy().tolist()[0]  # Obtener el embedding del texto
    collection.add(
        documents=[texto],
        metadatas=[metadata],
        ids=[texto_id],
        embeddings=[embedding]
    )

Agregando texto con ID: resena_007.txt y metadata: {'producto_nombre': 'Cafetera Negra', 'producto_id': '', 'puntaje': 5, 'sentimiento': 'positivo'}
Agregando texto con ID: resena_083.txt y metadata: {'producto_nombre': 'ventilador de torre', 'producto_id': 'P019', 'puntaje': 4, 'sentimiento': 'positivo'}
Agregando texto con ID: resena_019.txt y metadata: {'producto_nombre': 'horno', 'producto_id': 'P006', 'puntaje': 4, 'sentimiento': 'positivo'}
Agregando texto con ID: resena_076.txt y metadata: {'producto_nombre': 'Batidora de Pie', 'producto_id': '', 'puntaje': 5, 'sentimiento': 'positivo'}
Agregando texto con ID: resena_002.txt y metadata: {'producto_nombre': 'licuadora', 'producto_id': 'P001', 'puntaje': 3, 'sentimiento': 'neutral'}
Agregando texto con ID: resena_038.txt y metadata: {'producto_nombre': 'Batidora de Pie', 'producto_id': '', 'puntaje': 5, 'sentimiento': 'positivo'}
Agregando texto con ID: resena_032.txt y metadata: {'producto_nombre': 'freidora', 'producto_id': 'P01

In [3]:
print(collection.count())

561


### Test de consulta

In [4]:
consulta = "¿Como anda la cafetera negra?"
embedding_consulta = embed([consulta]).numpy().tolist()

results = collection.query(
    query_embeddings=embedding_consulta,  # Aquí pasamos el embedding de la consulta
    n_results=3  # Traemos los 3 resultados más cercanos
)

# imprimir cada resultado pero en formato tabular, utilizando pandas para mostrar los resultados en una tabla, una fila por cada resultado, y las columnas deben ser: IDs, Documento, Distancia y Metadata
result_ids = results["ids"][0] if results["ids"] else []
result_docs = results["documents"][0] if results["documents"] else []
result_distances = results["distances"][0] if results["distances"] else []
result_metadatas = results["metadatas"][0] if results["metadatas"] else []

df_resultados = pd.DataFrame({
    "IDs": result_ids,
    "Documento": result_docs,
    "Distancia": result_distances,
    "Metadata": result_metadatas
})
print(df_resultados)

              IDs                                          Documento  \
0  resena_010.txt  Perfecta! Soy muy cafetero y esta cafetera sup...   
1  resena_007.txt  La cafetera es UN SUEÑO! El molinillo integrad...   
2  resena_067.txt  La cafetera es hermosa y funcional! El café qu...   

   Distancia                                           Metadata  
0   0.950561  {'sentimiento': 'positivo', 'puntaje': 5, 'pro...  
1   0.982902  {'puntaje': 5, 'producto_id': '', 'producto_no...  
2   1.012703  {'producto_id': 'P003', 'sentimiento': 'positi...  


## 2. Cargar manuales

### Vectorizar los manuales

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

# Por cada archivo en  la carpeta: "data/manuales_productos/" cargar el contenido del archivo 
for filename in os.listdir("data/manuales_productos/"):
    if filename.endswith(".md"):
        with open(os.path.join("data/manuales_productos/", filename), "r") as file:
            contenido = file.read()
            print(f"Procesando el archivo {filename}:")

            # Dividir el contenido del archivo en fragmentos utilizando RecursiveCharacterTextSplitter            
            fragments = text_splitter.split_text(contenido)
            for i, fragment in enumerate(fragments):
                fragment_id = f"{filename}_fragment_{i}"
                embedding = embed([fragment]).numpy().tolist()[0]  # Obtener el embedding del fragmento
                metadata = {"source_file": filename, "fragment_index": i}
                print(f"Agregando fragmento con ID: {fragment_id} y metadata: {metadata}")
                collection.add(
                    documents=[fragment],
                    metadatas=[metadata],
                    ids=[fragment_id],
                    embeddings=[embedding]
                )

/home/gcasado0/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 302: Error loading CUDA libraries. GPU will not be used. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Procesando el archivo manual_coccion_saludable.md:
Agregando fragmento con ID: manual_coccion_saludable.md_fragment_0 y metadata: {'source_file': 'manual_coccion_saludable.md', 'fragment_index': 0}
Agregando fragmento con ID: manual_coccion_saludable.md_fragment_1 y metadata: {'source_file': 'manual_coccion_saludable.md', 'fragment_index': 1}
Agregando fragmento con ID: manual_coccion_saludable.md_fragment_2 y metadata: {'source_file': 'manual_coccion_saludable.md', 'fragment_index': 2}
Agregando fragmento con ID: manual_coccion_saludable.md_fragment_3 y metadata: {'source_file': 'manual_coccion_saludable.md', 'fragment_index': 3}
Agregando fragmento con ID: manual_coccion_saludable.md_fragment_4 y metadata: {'source_file': 'manual_coccion_saludable.md', 'fragment_index': 4}
Agregando fragmento con ID: manual_coccion_saludable.md_fragment_5 y metadata: {'source_file': 'manual_coccion_saludable.md', 'fragment_index': 5}
Agregando fragmento con ID: manual_coccion_saludable.md_fragment_6 

### Test de consulta

In [5]:
consulta = "Batidora se calienta excesivamente"
embedding_consulta = embed([consulta]).numpy().tolist()

results = collection.query(
    query_embeddings=embedding_consulta,  # Aquí pasamos el embedding de la consulta
    n_results=3  # Traemos los 3 resultados más cercanos
)

# imprimir cada resultado pero en formato tabular, utilizando pandas para mostrar los resultados en una tabla, una fila por cada resultado, y las columnas deben ser: IDs, Documento, Distancia y Metadata
result_ids = results["ids"][0] if results["ids"] else []
result_docs = results["documents"][0] if results["documents"] else []
result_distances = results["distances"][0] if results["distances"] else []
result_metadatas = results["metadatas"][0] if results["metadatas"] else []

df_resultados = pd.DataFrame({
    "IDs": result_ids,
    "Documento": result_docs,
    "Distancia": result_distances,
    "Metadata": result_metadatas
})
print(df_resultados)

                                IDs  \
0  manual_licuadoras.md_fragment_24   
1                    resena_016.txt   
2                    resena_060.txt   

                                           Documento  Distancia  \
0  ### Batidora se calienta excesivamente\n1. Dej...   1.221718   
1  La batidora está bien para uso ocasional. Si l...   1.278140   
2  Buen ventilador pero el timer no funciona bien...   1.331498   

                                            Metadata  
0  {'source_file': 'manual_licuadoras.md', 'fragm...  
1  {'puntaje': 3, 'producto_id': 'P005', 'sentimi...  
2  {'sentimiento': 'neutral', 'puntaje': 3, 'prod...  
